# Jak Wojtek uczy się chodzić

Wojtek to czworonożny robot, który zaczął jako **4BarBot** na Politechnice Wrocławskiej. Chodu nie ma zaprogramowanego: uczy się go metodą uczenia ze wzmocnieniem (RL) w symulacji, a wytrenowana sieć trafia potem na fizycznego robota. Ten notebook prowadzi przez tę drogę krok po kroku: robot w symulatorze MuJoCo, środowisko treningowe, trening, eksport polityki i na końcu porównanie Twojej polityki z tą, która jeździ na robocie. Każdy krok kończy się widokiem z MuJoCo.

## Robot

- 4 nogi, w każdej 3 silniki: odwodzenie biodra, biodro, kolano. Razem 12 silników sterowanych **pozycyjnie**: polityka podaje kąt docelowy, regulator PD w napędzie (na robocie MD80, w MuJoCo ten sam model serwa) zamienia go na moment.
- Nogi są czworobokami przegubowymi: poniżej silników łańcuch kinematyczny się zamyka. Za osobliwością kolana (ok. 3,2 rad) mechanizm może się przeskoczyć, dlatego cel kolana jest zawsze ograniczony.
- Masa 14 kg, wysokość stania ok. 0,125 m.
- Fizyka liczy się 250 razy na sekundę (krok 4 ms), polityka działa 50 razy na sekundę: jedna decyzja = 5 kroków fizyki.

## Skąd jest model

Wszystko leży w pakiecie ROS `ros/src/wojtek_description/`:

- `meshes/*.stl` — geometria ogniw robota z CAD-u;
- `urdf/*.urdf.xacro` — opis robota dla ROS-a (ten, którego używa prawdziwy robot i RViz);
- `mujoco/wojtek.xml` — ten sam robot zapisany w formacie MuJoCo (MJCF): ogniwa, przeguby, domknięcia czworoboków, siatki z `meshes/`, silniki i czujniki. Jest źródłem dla treningu;
- `mujoco/wojtek_mjx.xml` + `scene_mjx.xml` — wersja treningowa, **generowana** poleceniem `./training/run.sh build`. Skrypt bierze `wojtek.xml` i nanosi zmiany potrzebne w treningu: siatki przestają kolidować (stopy dostają kule, korpus prostopadłościan), korpus dostaje jawną masę, 12 silników momentowych staje się serwami PD, krok fizyki ustawiony na 4 ms. Tych plików nie edytuje się ręcznie; `scene_mjx.xml` dokłada podłogę, światło i kamerę śledzącą.

Notebook ładuje właśnie `scene_mjx.xml`, czyli dokładnie to, na czym trenuje polityka.

Środowisko Colab: **GPU** (Runtime → Change runtime type). Uruchamiaj komórki po kolei.

## Krok 0 — Instalacja

Klonuje repozytorium i instaluje `training/` (JAX, MuJoCo MJX, MJWarp, Brax). Jeśli następna komórka nie zaimportuje bibliotek, zrób Runtime → Restart session i uruchom od początku.

In [ ]:
import os, subprocess, sys
from pathlib import Path

if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")   # renderowanie bez ekranu; przed `import mujoco`

REPO_BRANCH = "Add-notebook-with-the-guidance-how-to-train-Wojtek"   # po scaleniu: "main"
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "training" / "run.sh").exists()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "w01-tek"
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, "https://github.com/machinekind/w01-tek.git", str(REPO_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "-q", "--ff-only"])   # ponowne uruchomienie: dociągnij zmiany
TRAINING = REPO_ROOT / "training"

try:
    import wojtek_rl, fast_simplification, trimesh  # noqa: F401
except ImportError:
    import tomllib
    lock = tomllib.loads((TRAINING / "uv.lock").read_text())
    mujoco_pin = next(p["version"] for p in lock["package"] if p["name"] == "mujoco")   # ta sama wersja co w locku
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(TRAINING), f"mujoco=={mujoco_pin}",
                          "trimesh", "fast-simplification"],   # dwa ostatnie: uproszczone siatki do renderowania
                         capture_output=True, text=True)
    if res.returncode:
        print(res.stdout[-1500:], res.stderr[-3000:])
        raise SystemExit("pip install nie powiodł się (patrz wyżej)")
    # Colab ma preinstalowany nowszy plugin JAX dla CUDA 13; obok jax 0.9.2 tylko generuje błędy.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax-cuda13-plugin", "jax-cuda13-pjrt"], capture_output=True)
    print("zainstalowano; jeśli następna komórka nie działa, zrestartuj sesję i uruchom od początku")
print(REPO_ROOT, "| python", sys.version.split()[0])

In [ ]:
import shutil
import mediapy as media
import mujoco
import numpy as np

if shutil.which("ffmpeg") is None:          # mediapy potrzebuje binarki ffmpeg
    import imageio_ffmpeg
    media.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

for p in (TRAINING, REPO_ROOT / "learning"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from wojtek_rl import paths
from lowpoly import render_model

# Colab nie ma OpenGL od NVIDII: MuJoCo renderuje programowo, a pełne siatki robota
# (600 tys. trójkątów) kosztują 0,8 s na klatkę. Rysujemy więc kopię modelu z uproszczonymi
# siatkami; fizyka liczy się na oryginale.
CACHE = TRAINING / "videos" / "guide" / "lowpoly"
rmodel = render_model(paths.SCENE_XML, CACHE)
rdata = mujoco.MjData(rmodel)
renderer = mujoco.Renderer(rmodel, height=360, width=480)

def frame(d, camera="track"):
    rdata.qpos[:] = d.qpos
    mujoco.mj_forward(rmodel, rdata)
    renderer.update_scene(rdata, camera=camera)
    return renderer.render().copy()

def show(frames, fps=25):
    media.show_video(np.asarray(frames), fps=fps, codec="h264")

print("mujoco", mujoco.__version__, "| model:", paths.SCENE_XML.relative_to(REPO_ROOT),
      f"| siatki do rysowania: {int(rmodel.mesh_facenum.sum()):,} trójkątów")

## Krok 1 — Wojtek stoi w MuJoCo

Robot startuje z zapisanej pozy `home` i trzyma jej kąty w serwach. Nic więcej: tak wygląda „zerowa akcja” polityki.

In [ ]:
model = mujoco.MjModel.from_xml_path(str(paths.SCENE_XML))
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
data.ctrl[:] = model.key("home").ctrl

frames = []
for k in range(int(4.0 / model.opt.timestep)):       # 4 s
    mujoco.mj_step(model, data)
    if k % 10 == 0:                                   # 25 klatek/s
        frames.append(frame(data))

print(f"serwa: {model.nu}, masa {sum(model.body_mass):.1f} kg, wysokość bazy {data.qpos[2]:.3f} m")
show(frames)

## Krok 2 — Polityka na początku treningu

Polityka to sieć neuronowa: na wejściu obserwacja, na wyjściu 12 przesunięć kątów względem pozy `home`. Obserwacja jest tym, co robot naprawdę mierzy: kąty i prędkości 12 przegubów, poprzednia akcja i **komenda** `[vx, vy, wz, wysokość]`, czyli wektor, w którym ma iść. Aktor ma warstwy 512-256-128, a w treningu do jego wyjścia dokłada się losowy szum, żeby próbował różnych ruchów.

Tak wygląda start PPO: te same wejścia, ta sama architektura, losowe wagi i szum. Komenda: 0,5 m/s do przodu. Sieć jeszcze nie wie, co komenda znaczy.

In [ ]:
KOMENDA = np.array([0.5, 0.0, 0.0, 0.125], np.float32)   # vx, vy, wz, wysokość stania
SEED = 0

model = mujoco.MjModel.from_xml_path(str(paths.SCENE_XML))
model.actuator_gainprm[:, 0], model.actuator_biasprm[:, 1], model.actuator_biasprm[:, 2] = 40.0, -40.0, -1.6   # serwo jak w treningu
model.actuator_forcerange[:] = [-9.0, 9.0]
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
HOME = model.key("home").ctrl.copy()
qadr = np.array([model.jnt_qposadr[j] for j in model.actuator_trnid[:, 0]])
vadr = np.array([model.jnt_dofadr[j] for j in model.actuator_trnid[:, 0]])
SCALE = np.tile([0.25, 0.5, 0.5], 4)                      # zakres akcji na przegub, rad
LOW, HIGH = model.actuator_ctrlrange.T.copy()
LOW[0::3], HIGH[0::3] = -0.44, 0.44                        # limit odwodzenia i kolana jak w treningu
HIGH[2::3] = np.minimum(HIGH[2::3], 3.15)

# Aktor jak w PPO na starcie: losowe wagi, wyjście = środek i rozrzut rozkładu akcji.
rng = np.random.default_rng(SEED)
sizes = [12 + 12 + 12 + 4, 512, 256, 128, 2 * 12]
W = [rng.uniform(-1, 1, (a, b)) * np.sqrt(3.0 / a) for a, b in zip(sizes[:-1], sizes[1:])]
def aktor(obs):
    x = obs
    for w in W[:-1]:
        x = x @ w; x = x / (1 + np.exp(-x))               # SiLU
    out = x @ W[-1]
    std = np.log1p(np.exp(out[12:])) + 1e-3               # softplus, jak w Brax
    return np.tanh(out[:12] + std * rng.standard_normal(12))

frames, last_act = [], np.zeros(12, np.float32)
x0 = data.qpos[0]
for i in range(200):                                       # 4 s przy 50 Hz
    obs = np.concatenate([data.qpos[qadr] - HOME, data.qvel[vadr], last_act, KOMENDA])
    last_act = aktor(obs).astype(np.float32)
    data.ctrl[:] = np.clip(HOME + last_act * SCALE, LOW, HIGH)
    for _ in range(5):
        mujoco.mj_step(model, data)
    if i % 2 == 0:
        frames.append(frame(data))
print(f"po 4 s: przebyte {data.qpos[0] - x0:+.2f} m w kierunku komendy (cel: +2.0 m), wysokość bazy {data.qpos[2]:.3f} m")
show(frames)